In [2]:
import json
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

In [ ]:
import pandas as pd

df = pd.read_csv('/Users/mariem/Desktop/docs_with_themes.csv')

# Filter only captions (not images)
captions = df[df['kind']=="caption"]

# Count occurrences by theme_id
theme_counts = captions['theme_id'].value_counts().head(10)
theme_counts


theme_id
1     50119
17    42959
58    41912
31    41156
43    40476
49    40263
61    39914
54    38205
76    37890
64    36910
Name: count, dtype: int64

In [5]:
top_ids = [1,17,58,31,43,49,61,54,76,64]

df[df["theme_id"].isin(top_ids)][["theme_id", "theme_label"]].drop_duplicates().sort_values("theme_id")


,theme_id,theme_label
90,1,"Told, Said, Don"
481,17,"Just, Like, Think"
2,31,"Think, Said, Just"
40,43,"Doctor, Effects, Doc"
21,49,"Ll, Let, Come"
828,54,"Just, Told, Don"
62,58,"Species, Extinct, Bird"
104,61,"Said, Just, Think"
237,64,"Gluten Free, Gluten, Mustard"
17,76,"Look Like, Look, Like"


In [7]:
theme_id = 76

captions_76 = df[(df["theme_id"] == theme_id) & (df["kind"] == "caption")]

captions_76.shape  # pour voir combien il y en a
captions_76[["contest_id", "doc_id", "text"]].head(10)


,contest_id,doc_id,text
17,510,caption_510_16,You probably know my column: Man on the Street.
94,510,caption_510_93,I'm a compulsive lier.
210,510,caption_510_209,Mom named me Matt and it just kinda' stuck.
214,510,caption_510_213,"Don't tread on me, I'm from New Hampshire."
258,510,caption_510_257,"Hi, I'm Matt!"
289,510,caption_510_288,Aren't we friends on Facebook?
293,510,caption_510_292,Has anybody seen me lately?
295,510,caption_510_294,My sleep number is 46th and Seventh.
403,510,caption_510_402,"I have a new perspective on chins, but that's ..."
470,510,caption_510_469,I used to count sheep.


In [9]:
# Themes mapping
theme_labels = {
    1: "Dialog & Banter Humor",
    17: "Everyday Observations",
    31: "Thoughts & Reflections",
    43: "Medical / Doctor Humor",
    49: "Colloquial Speech Humor",
    54: "Punchline Dialogues",
    58: "Animal & Nature Humor",
    61: "Opinions & Statements",
    64: "Food & Lifestyle Humor",
    76: "Identity & Self-Image Humor"
}

# Filter captions only
captions = df[df["kind"] == "caption"]

# Aggregate
top_ids = list(theme_labels.keys())
agg = (
    captions[captions['theme_id'].isin(top_ids)]
    .groupby('theme_id')[['funny','somewhat_funny','not_funny']]
    .sum()
    .reset_index()
)

# Build Flourish data
rows = []
for _, r in agg.iterrows():
    tid = int(r['theme_id'])
    label = theme_labels[tid]
    rows.append([label, "Funny", int(r['funny'])])
    rows.append([label, "Somewhat Funny", int(r['somewhat_funny'])])
    rows.append([label, "Not Funny", int(r['not_funny'])])

flourish_df = pd.DataFrame(rows, columns=["Source", "Target", "Value"])

# Save output
output_path = "/Users/mariem/Desktop/flourish_test.csv"
flourish_df.to_csv(output_path, index=False)

output_path

'/Users/mariem/Desktop/flourish_test.csv'